In [1]:
import math
import os
import yaml
import torch
import json
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

from daart.data import DataGenerator, compute_sequence_pad
from daart.transforms import ZScore
from daart_utils.data import DataHandler
from daart.models import Segmenter, GMDGM, RSLDSM

In [2]:
datas = {
    'fly': {
        'vids': [
            '2019_06_26_fly2',
            '2019_08_14_fly1',
            '2019_08_20_fly3',
            '2019_10_14_fly2',
            '2019_10_21_fly1',
        ],
        'parts': ['avg', 'still', 'walk', 'front_groom', 'back_groom', 'abdomen-move'],
        'sizes': [2,3,4,5],
        'ds_name': 'fly-5'
    },
    'oft': {
        'vids': [
            'OFT_39',
            'OFT_41',
            'OFT_43',
            'OFT_44',
            'OFT_49',
            'OFT_50',
            'OFT_51',
            'OFT_52',
            'OFT_54',
            'OFT_58',
        ],
        'parts': ['avg', 'supported', 'unsupported', 'grooming'],
        'sizes': [4,6,8,10],
        'ds_name': 'mouse-oft-aligned'
    } ,
    'ibl': {
        'vids': [
            'churchlandlab_CSHL045_2020-02-27-001',
            'cortexlab_KS020_2020-02-06-001',
            'hoferlab_SWC_043_2020-09-15-001',
            'mrsicflogellab_SWC_052_2020-10-22-001',
            'wittenlab_ibl_witten_27_2021-01-21-001',
        ],
        'parts': ['avg', 'still', 'move', 'wheel_turn', 'groom'],
        'sizes': [2,3,4,5],
        'ds_name': 'ibl'
    }, 
    'huga': {
        'vids': [
            'sess_06',
            'sess_08',
            'sess_11',
            'sess_13',
            'sess_17',
        ],
        'parts': ['avg', 'walking', 'running', 'going_up', 'going_down', 'sitting',
                     'sitting_down', 'standing_up', 'standing'],
        'sizes': [100,250,500,1000],
        'ds_name': 'huga'
    } 
}

input_dict = {
    'markers': 'm',
    'features-posvel': 'fp',
    'features-sturman': 'fs',
    'features-sturman-posvel': 'fspv'
}

data_path = '/home/bsb2144/daart_utils/data/'

In [22]:
ds = 'fly'
#ds = 'oft'
#ds='ibl'
#ds='huga'

input_type = 'markers'
#input_type = 'features-sturman'
#input_type = 'features-posvel'

model_names = [
    #'tcn',
    #'rsl',
    #'rsln',
    #'gm',
    #'gmnt'
]

save_names = [
    #'tcn_{}'.format(input_dict[input_type]),
    'rsl_{}'.format(input_dict[input_type]),
    #'rsln_{}'.format(input_dict[input_type]),
    #'gm_{}'.format(input_dict[input_type]),
    #'gmnt_{}'.format(input_dict[input_type]),
]


In [23]:
# note: for largest size v0 save states and latents
# loop over models
for mod_name, save_name in zip(model_names, save_names):
    # loop over data sizes
    sizes = datas[ds]['sizes']
    all_metrics = {}
    for size in sizes:
        # loop over versions
        size_metrics = []
        for v in range(5):
            # init model
            model_base = "/home/bsb2144/daart/results_daart/{}/multi-0/dtcn/".format(datas[ds]['ds_name'])
            if size == sizes[-1] and ds!='huga':
                model_dir = model_base + "{}-{}-good_sample-0_{}/version_{}".format(mod_name, size, input_type, v)
            else:
                model_dir = model_base + "{}-{}-good_sample-{}_{}/version_0".format(mod_name, size, v, input_type)
                
            model_file = os.path.join(model_dir, 'last_model.pt')
            arch_file = os.path.join(model_dir, 'hparams.yaml')
            with open(arch_file, 'rb') as f:
                hparams_new = yaml.safe_load(f)
            if hparams_new['model_class'] == 'segmenter':
                model_0 = Segmenter(hparams_new)
            elif hparams_new['model_class'] == 'rslds_marginal':
                model_0 = RSLDSM(hparams_new)
            elif hparams_new['model_class'] == 'gmdgm':
                model_0 = GMDGM(hparams_new)
            else:
                raise NotImplementedError('"%s" is an invalid model typr' % hparams_new['model_class'])
            model_0.load_state_dict(torch.load(
                model_file, map_location=lambda storage, loc: storage))

            model_0.to('cuda')
            model_0.eval()


            # loop over vids
            v_metrics = {
                'gt': [],
                'preds': []
            }
            for expt_id in datas[ds]['vids']:
                print(expt_id)
                # initialize data handler; point to correct base path
                handler = DataHandler(expt_id, base_path=os.path.join(data_path, datas[ds]['ds_name']))
                if input_type == 'markers':
                    markers_file = handler.get_marker_filepath()
                else:
                    markers_file = handler.get_feature_filepath(dirname=input_type)

                hand_labels_file = os.path.join(
                            "/home/bsb2144/daart/data/", datas[ds]['ds_name'], 'labels-hand', expt_id + '_labels.csv')

                # define data generator signals
                signals = ['markers', 'labels_strong']
                transforms = [ZScore(), None]
                paths = [markers_file, hand_labels_file]

                # build data generator
                data_gen_test = DataGenerator(
                    [expt_id], [signals], [transforms], [paths], device='cuda',#hparams['device'], 
                    batch_size=hparams_new['batch_size'], trial_splits='1;1;0;0', 
                    sequence_pad=hparams_new['sequence_pad'], sequence_length=hparams_new['sequence_length'],
                    input_type=hparams_new['input_type'])

                # load hand labels
                handler.load_hand_labels()
                states = np.argmax(handler.hand_labels.vals, axis=1)

                # compute predictions
                print('computing predictions for model 0...', end='')
                tmp = model_0.predict_labels(data_gen_test, return_scores=True)
                
                if 'tcn' in mod_name:
                    labels_pred = np.vstack(tmp['labels'][0])
                    lats = np.vstack(tmp['embedding'][0])
                elif 'rsl' in mod_name:
                    labels_pred = np.vstack(tmp['qy_x_probs'][0])
                    lats = np.vstack(tmp['qz_xy_mean'][0])
                else:
                    labels_pred = np.vstack(tmp['qy_x_probs'][0])
                    
                labels_model = np.argmax(labels_pred, axis=1)
                states = states[:len(labels_model)]
                
                # if largest size and v0 save preds and lats
                if 'tcn' in mod_name or 'rsl' in mod_name:
                    if size == sizes[-1] and v==0:
                        state_path = "/home/bsb2144/daart/metrics/{}/{}_{}_y_hat.npy".format(ds, expt_id, save_name)
                        lat_path = "/home/bsb2144/daart/metrics/{}/{}_{}_z_hat.npy".format(ds, expt_id, save_name)
                        np.save(state_path, labels_model)
                        np.save(lat_path, lats)
                
                v_metrics['gt'] += list(states)
                v_metrics['preds'] += list(labels_model)
                
            # agg results over all vids (one version)
            all_gt = np.array(v_metrics['gt'])
            all_preds = np.array(v_metrics['preds'])
            f1_by_class = f1_score(all_preds[all_gt>0], all_gt[all_gt>0], average=None)
            print('f1_by_class', f1_by_class)
            f1_avg = np.mean(f1_by_class)
            temp_res = {'avg': f1_avg}
            for part, score in zip(datas[ds]['parts'][1:], f1_by_class):
                temp_res[part] = score
            size_metrics.append(temp_res)
            print('temp_res',temp_res)
            
        # agg results over all versions (one size)
        size_temp = {}
        for part in datas[ds]['parts']:
            size_temp[part] = {}
            mean_temp = np.mean([arr[part] for arr in size_metrics])
            sd_temp = np.std([arr[part] for arr in size_metrics])/len(sizes)
            size_temp[part]['mean'] = mean_temp
            size_temp[part]['sd'] = sd_temp
            
        # save size metrics to res dict
        all_metrics[size] = size_temp
            
    # save results
    print(all_metrics)
    save_path = '/home/bsb2144/daart/metrics/{}/{}.json'.format(ds, save_name)
    with open(save_path, 'w') as f:
        json.dump(all_metrics, f)
        print('saved file to {}'.format(f))
    
    
    
    

sess_06
NZ:  0
computing predictions for model 0...sess_08
NZ:  0


/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


computing predictions for model 0...sess_11
NZ:  0
computing predictions for model 0...sess_13
NZ:  0
computing predictions for model 0...sess_17
NZ:  0
computing predictions for model 0...f1_by_class [0.81523504 0.81789605 0.65173143 0.7441831  0.90068518 0.69281139
 0.73246926 0.91902226]
temp_res {'avg': 0.7842542126340692, 'walking': 0.8152350363944214, 'running': 0.81789604989605, 'going_up': 0.6517314300819455, 'going_down': 0.7441830993055204, 'sitting': 0.9006851750871286, 'sitting_down': 0.6928113879003559, 'standing_up': 0.7324692588899967, 'standing': 0.9190222635171355}
sess_06
NZ:  0
computing predictions for model 0...sess_08
NZ:  0


/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


computing predictions for model 0...sess_11
NZ:  0
computing predictions for model 0...sess_13
NZ:  0
computing predictions for model 0...sess_17
NZ:  0
computing predictions for model 0...f1_by_class [0.7899684  0.61004389 0.7572297  0.66018554 0.96375603 0.82116364
 0.86197183 0.9178962 ]
temp_res {'avg': 0.7977769029269453, 'walking': 0.7899683967857343, 'running': 0.6100438861587785, 'going_up': 0.7572296965091853, 'going_down': 0.6601855381878257, 'sitting': 0.9637560333479597, 'sitting_down': 0.8211636428635268, 'standing_up': 0.8619718309859156, 'standing': 0.917896198576636}
sess_06
NZ:  0
computing predictions for model 0...sess_08
NZ:  0


...

sess_08
NZ:  0
computing predictions for model 0...sess_11
NZ:  0
computing predictions for model 0...sess_13
NZ:  0
computing predictions for model 0...sess_17
NZ:  0
computing predictions for model 0...f1_by_class [0.93040517 0.96802486 0.91223901 0.88661012 0.97744946 0.81848185
 0.89300104 0.92323321]
temp_res {'avg': 0.9136805903671807, 'walking': 0.9304051739351596, 'running': 0.9680248631952256, 'going_up': 0.91223901285907, 'going_down': 0.8866101166549083, 'sitting': 0.9774494556765163, 'sitting_down': 0.8184818481848184, 'standing_up': 0.8930010446202059, 'standing': 0.9232332078115422}
sess_06
NZ:  0
computing predictions for model 0...

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


sess_08
NZ:  0
computing predictions for model 0...sess_11
NZ:  0
computing predictions for model 0...sess_13
NZ:  0
computing predictions for model 0...sess_17
NZ:  0
computing predictions for model 0...f1_by_class [0.93060889 0.9721935  0.90323947 0.87577758 0.97366335 0.86540799
 0.92235106 0.92424057]
temp_res {'avg': 0.9209353008974105, 'walking': 0.9306088893051523, 'running': 0.9721934964954612, 'going_up': 0.9032394717169201, 'going_down': 0.8757775802970674, 'sitting': 0.9736633528838969, 'sitting_down': 0.8654079896280691, 'standing_up': 0.9223510595761696, 'standing': 0.9242405672765472}
sess_06
NZ:  0
computing predictions for model 0...

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


sess_08
NZ:  0
computing predictions for model 0...sess_11
NZ:  0
computing predictions for model 0...sess_13
NZ:  0
computing predictions for model 0...sess_17
NZ:  0
computing predictions for model 0...f1_by_class [0.92766638 0.89196928 0.89800848 0.90743512 0.97119935 0.83084299
 0.91156463 0.92094432]
temp_res {'avg': 0.9074538184720715, 'walking': 0.9276663809145462, 'running': 0.8919692803640846, 'going_up': 0.8980084787538204, 'going_down': 0.9074351217958925, 'sitting': 0.9711993509712894, 'sitting_down': 0.8308429884139034, 'standing_up': 0.9115646258503401, 'standing': 0.9209443207126949}
{100: {'avg': {'mean': 0.6945741680595099, 'sd': 0.018629391628987538}, 'walking': {'mean': 0.7570203294787772, 'sd': 0.012114153358718904}, 'running': {'mean': 0.690978724200155, 'sd': 0.02617048951836056}, 'going_up': {'mean': 0.6666822474851577, 'sd': 0.0063448035248779}, 'going_down': {'mean': 0.564061362805137, 'sd': 0.03395032856647795}, 'sitting': {'mean': 0.8535070271763232, 'sd': 0.

In [24]:
print('done')

done
